# 🦋 Insect Anatomy YOLO-Seg (v2: High-Res + Data Augmentation)

This notebook trains a higher-capacity YOLO instance segmentation model on your insect annotations.

### What's New in v2:
1. **Bigger Model (`yolov8s-seg` / `yolo11s-seg`):** ~11M parameters (3.5x capacity of nano) to capture complex wing veins, pectinate antennae, and legs on textured backgrounds.
2. **Higher Resolution (`imgsz=1024`):** Retains delicate sub-millimeter appendages that get lost at 640px.
3. **Domain Adaptation Augmentations:** Aggressive HSV lighting jitter (sunlight vs shade vs flash), scale variations (+/- 50% for macro vs full insect), random rotation (+/- 30°), and Mosaic/Mixup.
4. **Verified iNat Samples Mixed In:** Learns directly from real iNaturalist outdoor photos.

### 1. Check Hardware & Sync Latest Annotations

In [ ]:
import torch
import os

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🚀 PyTorch version: {torch.__version__}")
print(f"⚡ Compute device:  {device} ({'GPU: ' + torch.cuda.get_device_name(0) if device != 'cpu' else 'CPU multi-core'})")

# Sync any recently verified iNat samples into the training dataset
!python cluster_yolo_training/update_dataset.py

### 2. Verify Dataset
Checks images and classes in `dataset.yaml`.

In [ ]:
import glob
import os
import yaml

dataset_dir = os.path.abspath('cluster_yolo_training/dataset' if os.path.exists('cluster_yolo_training/dataset') else 'dataset')
yaml_path = os.path.join(dataset_dir, 'dataset.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(f'''path: {dataset_dir}
train: images/train
val: images/val

names:
  0: full-body
  1: wing
  2: leg
  3: antenna
  4: head
  5: body
''')

with open(yaml_path) as f:
    dataset_cfg = yaml.safe_load(f)

print("Classes:", dataset_cfg.get('names'))
train_imgs = glob.glob(os.path.join(dataset_dir, 'images/train/*.jpg'))
val_imgs = glob.glob(os.path.join(dataset_dir, 'images/val/*.jpg'))
print(f"Train samples: {len(train_imgs)} (includes colleague field + downscaled iNat!)")
print(f"Val samples:   {len(val_imgs)}")

### 2b. Preview Data Augmentations (Field & Macro Domain Adaptations)
Visualizes simulated training augmentations (HSV lighting/flash shifts, rotation, zoom, blur, and sensor grain) on both a field camera trap image and an iNaturalist sample.
If a training run has already started, this also displays YOLO's native training batch mosaic.

In [ ]:
import os
import glob
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

dataset_dir = os.path.abspath("cluster_yolo_training/dataset" if os.path.exists("cluster_yolo_training/dataset") else "dataset")
train_imgs = glob.glob(os.path.join(dataset_dir, "images/train/*.jpg"))

field_imgs = [f for f in train_imgs if "ind_" in os.path.basename(f)]
inat_imgs = [f for f in train_imgs if "ind_" not in os.path.basename(f)]

samples_to_show = []
if field_imgs:
    samples_to_show.append(("Field Camera Trap", random.choice(field_imgs)))
if inat_imgs:
    samples_to_show.append(("Degraded iNaturalist", random.choice(inat_imgs)))

def apply_simulated_augs(bgr):
    """Simulates training augmentations (HSV lighting, rotation/zoom, grain)."""
    h, w = bgr.shape[:2]
    # 1. Lighting / HSV shift (flash / low light)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-10, 10)) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.6, 1.4), 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.6, 1.4), 0, 255)
    img_hsv = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # 2. Rotation & Zoom
    angle = random.uniform(-30, 30)
    scale = random.uniform(0.7, 1.3)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    img_geom = cv2.warpAffine(bgr, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 3. Field blur + sensor grain
    ksize = random.choice([3, 5])
    img_blur = cv2.GaussianBlur(img_geom, (ksize, ksize), 0)
    noise = np.random.normal(0, random.uniform(6, 14), bgr.shape).astype(np.float32)
    img_combined = np.clip(img_blur.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return img_hsv, img_geom, img_combined

# Plot augmentations grid
fig, axes = plt.subplots(len(samples_to_show), 4, figsize=(16, 4 * len(samples_to_show)))
if len(samples_to_show) == 1:
    axes = np.expand_dims(axes, 0)

col_titles = [
    "1. Original Dataset Image",
    "2. HSV Lighting Shift (Flash/Shadow)",
    "3. Rotation (±30°) & Zoom (±50%)",
    "4. Combined (Rotation + Noise + Blur)"
]

for row_idx, (tag, img_path) in enumerate(samples_to_show):
    bgr = cv2.imread(img_path)
    aug1, aug2, aug3 = apply_simulated_augs(bgr)
    
    imgs = [bgr, aug1, aug2, aug3]
    for col_idx, im in enumerate(imgs):
        ax = axes[row_idx, col_idx]
        rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        ax.imshow(rgb)
        ax.set_title(f"{tag}
{col_titles[col_idx]}", fontsize=10)
        ax.axis("off")

plt.tight_layout()
plt.show()

# If training has already run, show native YOLO mosaic batch
batch_mosaic = "runs/insect_anatomy_v2/train_batch0.jpg"
if os.path.exists(batch_mosaic):
    print("🎨 Native YOLO Multi-Image Training Mosaic:")
    display(Image(batch_mosaic, width=700))


### 3. Start Training (v2 with High-Res & Augmentations)
- Model: `yolov8s-seg.pt` (Small)
- Resolution: `1024` (preserves fine antenna and leg textures)
- Epochs: `150` with Cosine Annealing learning rate

In [ ]:
from ultralytics import YOLO

# Load small model with 3.5x capacity of nano
model = YOLO('yolov8s-seg.pt')

batch_size = 16 if torch.cuda.is_available() else 4
epochs = 150

results = model.train(
    data=yaml_path,
    epochs=epochs,
    batch=batch_size,
    imgsz=640,
    device=device,
    patience=30,
    cos_lr=True,            # Cosine learning rate schedule
    # --- Domain Augmentations for iNat & Macro Lighting ---
    hsv_h=0.015,            # Hue jitter
    hsv_s=0.7,              # Saturation jitter (sun vs shade)
    hsv_v=0.4,              # Value / brightness jitter (shadows vs flash)
    degrees=30.0,           # Random rotations up to 30 deg
    scale=0.5,              # Multi-scale zoom (+/- 50%)
    fliplr=0.5,             # Horizontal flips
    flipud=0.5,             # Vertical flips
    mosaic=1.0,             # Multi-image context mosaic
    mixup=0.15,             # Blend overlapping structures
    close_mosaic=15,        # Turn off mosaic for clean boundary learning
    project='runs',
    name='insect_anatomy_v2',
    exist_ok=True,
    save=True,
    plots=True
)

### 4. Inspect Results & Loss Curves

In [ ]:
from IPython.display import Image, display

results_png = 'runs/insect_anatomy_v2/results.png'
if os.path.exists(results_png):
    display(Image(results_png))
else:
    print("Results plot not found yet.")

### 4b. Visual Segmentation Results (Field & iNat Test Set)
Runs inference on validation images using  and displays predicted segmentation masks, bounding boxes, and anatomical part counts inline.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import glob, os, random, cv2
from collections import Counter

os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

# 1. Load trained best weights
weights_candidates = [
    "cluster_yolo_training/weights/best.pt",
    "runs/insect_anatomy_v2/weights/best.pt",
    "weights/best.pt"
]
best_weights = next((w for w in weights_candidates if os.path.exists(w)), None)
assert best_weights is not None, "❌ Trained weights (best.pt) not found. Run training first!"
print(f"✔ Loading model from: {best_weights}")
val_model = YOLO(best_weights)

# 2. Select a representative sample of field camera trap + iNat validation images
val_dir = "dataset/images/val" if os.path.exists("dataset/images/val") else "cluster_yolo_training/dataset/images/val"
val_all = sorted(glob.glob(os.path.join(val_dir, "*.jpg")))

val_field = [f for f in val_all if "ind_" in os.path.basename(f)]
val_inat = [f for f in val_all if "ind_" not in os.path.basename(f)]

# Pick 4 field images and 2 iNat images (or whatever is available)
selected = []
if val_field:
    selected.extend(val_field[:4])
if val_inat:
    selected.extend(val_inat[:2])
if not selected:
    selected = val_all[:6]

print(f"🔍 Running inference on {len(selected)} validation images (Conf >= 0.25)...")
results = val_model.predict(source=selected, conf=0.25, iou=0.45, max_det=30, verbose=False)

# 3. Display segmentation results directly in notebook
num_imgs = len(results)
cols = 3
rows = (num_imgs + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for idx, (r, ax) in enumerate(zip(results, axes)):
    # r.plot() draws colored polygon masks, boxes, labels & confidences
    annotated_rgb = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    
    # Count detected classes
    counts = Counter()
    if r.boxes and r.boxes.cls is not None:
        for c in r.boxes.cls.cpu().numpy():
            cls_name = r.names.get(int(c), str(int(c)))
            counts[cls_name] += 1
    
    parts_summary = ", ".join(f"{cnt} {cls}" for cls, cnt in counts.items()) if counts else "No detections"
    fname = os.path.basename(r.path)
    source_tag = "📷 FIELD" if "ind_" in fname else "🌿 iNAT"
    
    ax.set_title(f"{source_tag}: {fname}
{parts_summary}", fontsize=11, fontweight="bold")
    ax.axis("off")

# Hide any extra empty subplots
for j in range(idx + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


### 5. Save & Export Weights

In [ ]:
import shutil

os.makedirs('cluster_yolo_training/weights', exist_ok=True)
best_src = 'runs/insect_anatomy_v2/weights/best.pt'
best_dst = 'cluster_yolo_training/weights/best.pt'

if os.path.exists(best_src):
    shutil.copy2(best_src, best_dst)
    print(f"✔ Saved best model to: {best_dst}")
    
    best_model = YOLO(best_dst)
    onnx_path = best_model.export(format='onnx', imgsz=1024)
    print(f"✔ Exported to ONNX: {onnx_path}")

### 6. Re-predict Remaining Active Backlog (AI-assisted Pre-annotation)
Runs the improved model to generate sharper masks on the remaining images.

In [ ]:
!python cluster_yolo_training/predict_and_annotate.py --weights cluster_yolo_training/weights/best.pt --target data/active_annotation_set --overwrite-ai